# SummarizationMiddleware

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

chat_model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", )

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool(parse_docstring=True)
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool(parse_docstring=True)
def read_email_tool(email_id: str) -> str:
    """
    通过邮件ID读取内容的伪函数

    Args:
        email_id: email id
    """
    return f"邮件ID：{email_id}\n是空的"


@tool(parse_docstring=True)
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """
    发送邮件伪函数

    Args:
        recipient: 收件人
        subject：邮件标题
        body：邮件内容
    """
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model=chat_model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断啦"
                },
            },
            description_prefix="中断啦"
        ),
    ],
)

config = {"configurable": {"thread_id": "2"}}

# 第一次调用：会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气"
                                 "查询今日新闻"
                                 "查看ID为 'sk2131421' 的邮件内容，"
                                 "向15641685664@qq.com发送邮件，标题是'哈哈哈'， 内容是：'你好啊'"
                                 "同时做这四件事")
        ]
    },
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()
# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
rprint(interrupts)

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'， 内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='7fd75f4e-6bb8-4a0d-9c09-541706c5c1e2'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqawqtTJRG48MvDuLj-MGbbHTsfGB4Yxbnt4jJIj1mikzdgOBIXOoBZlqy-GwjYUPTrBIxoN5JPhOInOQHGJrupve80Yd_7JXQJn-2E-hiIm
EGA76itDE-dLhMf8KwACkWsKrYt8AA5NtKQthoPhn3DAALjiXF2Bx7yfi6yHFfqXWbefdUDQ4ozMWaewDRLwu_ugDSFf98LAitqg9zPTYjfm0Np2PLP
EDEvvH6q-XpTvqzkhGm09mkR7D5VsdFM4gpwwsBTW53gbLbJu-TNPSZOXU96oo56xoQMOekpTx3sqAIoQE7_hZwVWvIP2Gp1RMDTpkHxJxzRVYI_2UL
0sJdGWfJqpfcJkHIuQTUPHOJg9BvoEGbiqtBeUiT7BFP18I7AaEXdxlhFz-ipqIODSmYTFh0aEw7GAtE_GBPJkDCtWZ53tJM3Ty4y_ejHc0oOzHBot2
Uv32qe5rHlJQq677iU63uuzlJCfOQJOwoTEd0ZWwcm3PSl5Vu3Grlu0MFN1hLspUgZXk8KPyuZ350nk6X24N6loZarrW0L6YGVoD_i7ON6E3bgbNVVv
2IfUwJ5v6PutYdqPTE5x8g7ixGpYjD2tS-huR13wLGwHE324YQaAKMr8GvZUL2mUHT8KwRRE2jMb1dtpXwffWV9VSHldQy9OmyUfiFza-IAdDNGRABy
pbqulMjX1LWw990UQeVN7kh6WmsYf7lLfpGl4J-dm1zsEQpxiFHhC0AMcu33NVOYgm87P70v4h0db7mJFNRxZsdt94oltTqVJYRE6uouUubtz4fGjRf
bjvk3CEq8dqsGlXgiEq0d6eCkV1aPmvQhqMlGVa77Rdhfo1GcAWyHFYe-B6QAVXTbnPmz91y1hayYM_fHlEZhGmWo4efuB3JrUZIPAzRPQ4x5vySchS
3on91Y0CPd4Q0Rc05bMH1YQ9Azaf2ExK8U1OdN5nP_IJx35NNYmdzjbNqPthzEhOtkSDIkALjWl0OGvUCdI-I19Bn4drd9_1gQDh3ItZEkD65hGut7B
OjGo5hQmrYx_dXvzzFehMllESS7MCcJVtkfTz5vZgT6w8=',
                        'id': 'rs_0d2f2ce5623abec8016a6b0aac441c81928eb490f9c84709ee',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785399979-dWcZfwCHMK1tboQg96YS',
                'created': 1785399979,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fb221-ab7d-71d3-bfd8-cb57bdb0285f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_qK1H7e8V5MgI6XY2JGHS8a0G',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_NN3suYkRn7cBpMFh7Ad73fcl', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_snDV9JLC6s8HX5IgnnDBdFJB',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_j8oohSh03o2C9HBTrrpBJwsc',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 206,
                'output_tokens': 131,
                'total_tokens': 337,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 29}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\n

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'， 内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_qK1H7e8V5MgI6XY2JGHS8a0G)
 Call ID: call_qK1H7e8V5MgI6XY2JGHS8a0G
  Args:
    city: 北京
    is_forcast: False
  get_news (call_NN3suYkRn7cBpMFh7Ad73fcl)
 Call ID: call_NN3suYkRn7cBpMFh7Ad73fcl
  Args:
  read_email_tool (call_snDV9JLC6s8HX5IgnnDBdFJB)
 Call ID: call_snDV9JLC6s8HX5IgnnDBdFJB
  Args:
    email_id: sk2131421
  send_email_tool (call_j8oohSh03o2C9HBTrrpBJwsc)
 Call ID: call_j8oohSh03o2C9HBTrrpBJwsc
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
========== interrupts ==========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'description': '发送邮件中断啦'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='a6ab1cf3cf0e64d3e3e5e73fb4e0cc37'
    )
]

In [13]:
from langgraph.types import Command

# 如果有中断，说明进入人在环了
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "中国上海市", "is_forcast": True}
    }
}
news_decision = {
    "type": "approve",
}
send_email_decision = {
    "type": "approve"
}
decisions = {
    "decisions": []
}
# 决策的顺序必须和返回的中断请求顺序一致
interrupts = response.get("__interrupt__", [])

if interrupts:
    action_requests = interrupts[0].value.get("action_requests", [])

    for action_request in action_requests:
        if action_request["name"] == "get_weather":
            decisions["decisions"].append(weather_decision)
        if action_request["name"] == "get_news":
            decisions["decisions"].append(news_decision)
        if action_request["name"] == "send_email_tool":
            decisions["decisions"].append(send_email_decision)

    # 审批通过
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config=config,  # 必须是同一个 thread_id
    )

    print("==== 审批后继续执行 ====")
    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
==== 审批后继续执行 ====
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'， 内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_qK1H7e8V5MgI6XY2JGHS8a0G)
 Call ID: call_qK1H7e8V5MgI6XY2JGHS8a0G
  Args:
    city: 中国上海市
    is_forcast: True
  get_news (call_NN3suYkRn7cBpMFh7Ad73fcl)
 Call ID: call_NN3suYkRn7cBpMFh7Ad73fcl
  Args:
  read_email_tool (call_snDV9JLC6s8HX5IgnnDBdFJB)
 Call ID: call_snDV9JLC6s8HX5IgnnDBdFJB
  Args:
    email_id: sk2131421
  send_email_tool (call_j8oohSh03o2C9HBTrrpBJwsc)
 Call ID: call_j8oohSh03o2C9HBTrrpBJwsc
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

中国上海市今天天气不错
明天下雨
================================= Tool Message ===========================